# Adaptive Shield Weekly Report

自動化產出 Configuration Drift 與 Integration Failure 週報。

**流程**: API 拉數據 → 攤平 → 合併 → 匯出 XLSX

In [ ]:
# Cell 1: Setup & Configuration
import os
import logging
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
import pandas as pd

from as_client import AdaptiveShieldClient, AuthError

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# Load .env
load_dotenv()

AS_API_KEY = os.getenv("AS_API_KEY")
AS_BASE_URL = os.getenv("AS_BASE_URL", "https://api.adaptive-shield.com")
AS_ACCOUNT_IDS = [
    x.strip()
    for x in os.getenv("AS_ACCOUNT_IDS", "").split(",")
    if x.strip()
]
DAYS_BACK = int(os.getenv("DAYS_BACK", "3"))
SNOW_ENABLED = os.getenv("SNOW_ENABLED", "false").lower() == "true"

if not AS_API_KEY:
    raise ValueError("AS_API_KEY is required. Set it in .env file.")

# Output directory
today = datetime.now().strftime("%Y-%m-%d")
output_dir = Path("output") / today
output_dir.mkdir(parents=True, exist_ok=True)

client = AdaptiveShieldClient(AS_API_KEY, AS_BASE_URL)

print(f"Config loaded: base_url={AS_BASE_URL}, days_back={DAYS_BACK}")
print(f"Account IDs: {AS_ACCOUNT_IDS or '(auto-discover)'}")
print(f"Output dir: {output_dir}")

In [ ]:
# Cell 2: Data Collection

# Auto-discover accounts if not configured
if not AS_ACCOUNT_IDS:
    accounts = client.get_accounts()
    AS_ACCOUNT_IDS = [a.get("id") or a.get("account_id") for a in accounts]
    logger.info("Auto-discovered %d accounts: %s", len(AS_ACCOUNT_IDS), AS_ACCOUNT_IDS)

all_alert_records = []
all_integration_records = []

for account_id in AS_ACCOUNT_IDS:
    logger.info("Processing account: %s", account_id)

    # ── Alerts ──
    try:
        alerts = client.get_alerts(account_id, days_back=DAYS_BACK)
        logger.info("  Found %d alerts", len(alerts))

        # Debug: log first alert keys for API field validation
        if alerts:
            logger.info("  Alert keys: %s", list(alerts[0].keys()))
    except AuthError:
        raise
    except Exception as e:
        logger.error("  Failed to fetch alerts: %s", e)
        alerts = []

    for alert in alerts:
        try:
            # Extract check ID
            check_id = (
                alert.get("security_check_id")
                or alert.get("check_id")
                or alert.get("security_check", {}).get("id")
            )

            # Enrich with security check details (cached)
            check = {}
            if check_id:
                check = client.get_security_check(account_id, check_id)
                if check:
                    logger.debug("  Check keys: %s", list(check.keys()))

            # Resolve affected entities
            affected_count = (
                alert.get("new_affected_count")
                or alert.get("affected_count")
                or alert.get("affected_entities_count")
                or 0
            )
            affected_diff = alert.get("affected_diff") or alert.get("affected_entities_diff")
            affected_detail = ""

            if affected_count == 0 and not affected_diff:
                affected_detail = "Global"
                affected_count = "Global"
            elif affected_diff:
                # Use embedded diff data
                if isinstance(affected_diff, list):
                    names = [
                        e.get("name") or e.get("entity_name") or str(e)
                        for e in affected_diff
                    ]
                    affected_detail = ", ".join(names)
                else:
                    affected_detail = str(affected_diff)
            elif check_id:
                # Fetch from API
                try:
                    entities = client.get_affected_entities(account_id, check_id)
                    if entities:
                        names = [
                            e.get("name") or e.get("entity_name") or str(e)
                            for e in entities
                        ]
                        affected_detail = ", ".join(names)
                    else:
                        affected_detail = f"{affected_count} entities"
                except Exception:
                    affected_detail = f"{affected_count} entities"

            # Extract integration info
            integration = alert.get("integration") or {}

            record = {
                "change_datetime": (
                    alert.get("timestamp")
                    or alert.get("created_at")
                    or alert.get("date")
                    or ""
                ),
                "security_check_name": (
                    check.get("name")
                    or check.get("check_name")
                    or alert.get("security_check_name")
                    or ""
                ),
                "security_check_details": (
                    check.get("description")
                    or check.get("details")
                    or ""
                ),
                "remediation_steps": (
                    check.get("remediation")
                    or check.get("remediation_steps")
                    or ""
                ),
                "impact_level": (
                    check.get("severity")
                    or check.get("impact_level")
                    or alert.get("severity")
                    or ""
                ),
                "current_status": "Degraded",
                "affected_entities_count": affected_count,
                "affected_entities_detail": affected_detail,
                "integration_name": (
                    integration.get("name")
                    or alert.get("integration_name")
                    or ""
                ),
                "integration_alias": (
                    integration.get("alias")
                    or alert.get("integration_alias")
                    or ""
                ),
            }
            all_alert_records.append(record)

        except Exception as e:
            logger.error("  Failed to process alert %s: %s", alert.get("id", "?"), e)
            continue

    # ── Integration Failures ──
    try:
        failed_integrations = client.get_integrations(account_id)
        logger.info("  Found %d integration failures", len(failed_integrations))

        for intg in failed_integrations:
            all_integration_records.append({
                "integration_name": intg.get("name") or "",
                "integration_alias": intg.get("alias") or "",
                "status": intg.get("status") or "",
                "issues": str(intg.get("issues") or intg.get("errors") or ""),
                "last_sync": intg.get("last_sync") or intg.get("last_updated") or "",
            })
    except AuthError:
        raise
    except Exception as e:
        logger.error("  Failed to fetch integrations: %s", e)

print(f"\nCollection complete: {len(all_alert_records)} alerts, {len(all_integration_records)} integration failures")

In [ ]:
# Cell 3: Transform & Display

df_alerts = pd.DataFrame(all_alert_records)
df_integrations = pd.DataFrame(all_integration_records)

# Sort alerts by datetime descending
if not df_alerts.empty and "change_datetime" in df_alerts.columns:
    df_alerts["change_datetime"] = pd.to_datetime(
        df_alerts["change_datetime"], errors="coerce"
    )
    df_alerts = df_alerts.sort_values("change_datetime", ascending=False).reset_index(drop=True)

# Summary stats
print("=" * 60)
print(f"ADAPTIVE SHIELD WEEKLY REPORT — {today}")
print(f"Period: last {DAYS_BACK} days")
print("=" * 60)

if not df_alerts.empty:
    print(f"\nTotal config drift alerts: {len(df_alerts)}")
    if "impact_level" in df_alerts.columns:
        severity_counts = df_alerts["impact_level"].value_counts()
        for level, count in severity_counts.items():
            print(f"  {level}: {count}")
else:
    print("\nNo config drift alerts found.")

if not df_integrations.empty:
    print(f"\nIntegration failures: {len(df_integrations)}")
else:
    print("\nNo integration failures found.")

print()

# Display tables
if not df_alerts.empty:
    print("\n--- Config Drift Alerts ---")
    display(df_alerts)

if not df_integrations.empty:
    print("\n--- Integration Failures ---")
    display(df_integrations)

In [ ]:
# Cell 4: Export to XLSX

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
xlsx_path = output_dir / f"AS_Weekly_Report_{timestamp}.xlsx"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    # Config Drift sheet
    if not df_alerts.empty:
        df_alerts.to_excel(writer, sheet_name="Config Drift", index=False)
    else:
        pd.DataFrame({"Info": ["No config drift alerts found"]}).to_excel(
            writer, sheet_name="Config Drift", index=False
        )

    # Integration Failures sheet
    if not df_integrations.empty:
        df_integrations.to_excel(
            writer, sheet_name="Integration Failures", index=False
        )
    else:
        pd.DataFrame({"Info": ["No integration failures found"]}).to_excel(
            writer, sheet_name="Integration Failures", index=False
        )

    # Summary sheet
    summary_data = {
        "Metric": [
            "Report Date",
            "Period (days)",
            "Total Config Drift Alerts",
            "Total Integration Failures",
        ],
        "Value": [
            today,
            DAYS_BACK,
            len(df_alerts),
            len(df_integrations),
        ],
    }

    # Add severity breakdown
    if not df_alerts.empty and "impact_level" in df_alerts.columns:
        for level, count in df_alerts["impact_level"].value_counts().items():
            summary_data["Metric"].append(f"  {level}")
            summary_data["Value"].append(count)

    pd.DataFrame(summary_data).to_excel(
        writer, sheet_name="Summary", index=False
    )

print(f"Report exported: {xlsx_path}")